# Residual U-Net.

In [1]:
from pathlib import Path
import random, time
from dataclasses import dataclass
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", DEVICE)

PyTorch: 2.11.0+cu128
CUDA available: True
Device: cuda


In [ ]:
@dataclass
class CFG:
    DATA_ROOT: Path = Path()
    INPUT_DIRNAME: str = "inputs"
    MASK_DIRNAME: str = "masks"
    SPLIT_CSV: str = ""

    BATCH_SIZE: int = 8
    NUM_WORKERS: int = 0
    PIN_MEMORY: bool = torch.cuda.is_available()

    SEED: int = 30
    EPOCHS: int = 30
    LR: float = 1e-4
    WEIGHT_DECAY: float = 1e-5
    MASK_THRESHOLD: float = 0.5

cfg = CFG()

INPUT_DIR = cfg.DATA_ROOT / cfg.INPUT_DIRNAME
MASK_DIR = cfg.DATA_ROOT / cfg.MASK_DIRNAME
SPLIT_PATH = cfg.DATA_ROOT / cfg.SPLIT_CSV

In [ ]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(cfg.SEED)

# Load split table and verify case files

| case_id | split | % |
|---|---|---|
| case_001 | train | 70 |
| case_002 | val | 10 |
| case_003 | test | 20 |

In [ ]:
assert SPLIT_PATH.exists(), f"Missing split CSV: {SPLIT_PATH}"

splits_df = pd.read_csv(SPLIT_PATH)
required_cols = {"case_id", "split"}
assert required_cols.issubset(splits_df.columns), f"Missing columns: {required_cols - set(splits_df.columns)}"

splits_df["case_id"] = splits_df["case_id"].astype(str)
splits_df["split"] = splits_df["split"].str.lower().str.strip()

valid_splits = {"train", "val", "test"}
assert set(splits_df["split"].unique()).issubset(valid_splits), "Split must be train/val/test."

print(splits_df["split"].value_counts())
splits_df.head()


In [ ]:
def case_path(case_id: str, folder: Path) -> Path:
    return folder / f"{case_id}.npy"

manifest = []
for _, row in splits_df.iterrows():
    case_id = row["case_id"]
    input_path = case_path(case_id, INPUT_DIR)
    mask_path = case_path(case_id, MASK_DIR)

    manifest.append({
        "case_id": case_id,
        "split": row["split"],
        "input_path": str(input_path),
        "mask_path": str(mask_path),
        "input_exists": input_path.exists(),
        "mask_exists": mask_path.exists(),
    })

manifest_df = pd.DataFrame(manifest)

print("Missing input files:", (~manifest_df["input_exists"]).sum())
print("Missing mask files:", (~manifest_df["mask_exists"]).sum())

assert manifest_df["input_exists"].all(), "Some input files are missing."
assert manifest_df["mask_exists"].all(), "Some mask files are missing."

manifest_df.head()


## Strict array validation

Expected:
- input: `(N, 3, H, W)`
- mask: `(N, 1, H, W)`


In [ ]:
case_stats = []

for _, row in manifest_df.iterrows():
    x = np.load(row["input_path"], mmap_mode="r")
    y = np.load(row["mask_path"], mmap_mode="r")

    assert x.ndim == 4, f"{row['case_id']}: input shape must be (N,3,H,W), got {x.shape}"
    assert y.ndim == 4, f"{row['case_id']}: mask shape must be (N,1,H,W), got {y.shape}"
    assert x.shape[0] == y.shape[0], f"{row['case_id']}: number of samples mismatch."
    assert x.shape[1] == 3, f"{row['case_id']}: expected 3 input channels, got {x.shape[1]}"
    assert y.shape[1] == 1, f"{row['case_id']}: expected 1 mask channel, got {y.shape[1]}"
    assert x.shape[-2:] == y.shape[-2:], f"{row['case_id']}: spatial mismatch."

    positive_per_sample = np.sum(y > 0, axis=(1,2,3)) > 0

    case_stats.append({
        "case_id": row["case_id"],
        "split": row["split"],
        "num_samples": int(x.shape[0]),
        "height": int(x.shape[2]),
        "width": int(x.shape[3]),
        "positive_samples": int(positive_per_sample.sum()),
        "positive_pixels": int(np.sum(y > 0)),
        "x_min": float(np.min(x)),
        "x_max": float(np.max(x)),
    })

case_stats_df = pd.DataFrame(case_stats)
print("All cases passed validation.")
case_stats_df.head()


In [ ]:
summary_df = (
    case_stats_df
    .groupby("split")
    .agg(
        cases=("case_id", "nunique"),
        total_samples=("num_samples", "sum"),
        positive_samples=("positive_samples", "sum"),
        positive_pixels=("positive_pixels", "sum"),
    )
    .reset_index()
)

summary_df["positive_sample_ratio"] = summary_df["positive_samples"] / summary_df["total_samples"]
summary_df


## Visual sanity check

Must inspect at least several positive samples before training. If mask alignment is wrong, stop.

In [ ]:
def load_positive_sample(case_id: str):
    row = manifest_df[manifest_df["case_id"] == case_id].iloc[0]
    x = np.load(row["input_path"], mmap_mode="r")
    y = np.load(row["mask_path"], mmap_mode="r")

    positive_idx = np.where(np.sum(y > 0, axis=(1,2,3)) > 0)[0]
    if len(positive_idx) == 0:
        return None

    idx = int(positive_idx[0])
    return np.array(x[idx]), np.array(y[idx]), idx

positive_cases = case_stats_df[case_stats_df["positive_samples"] > 0]["case_id"].tolist()
assert positive_cases, "No positive masks found."

case_id = positive_cases[0]
x_sample, y_sample, sample_idx = load_positive_sample(case_id)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for i, title in enumerate(["slice k-1", "slice k", "slice k+1"]):
    axes[i].imshow(x_sample[i], cmap="gray")
    axes[i].set_title(title)
    axes[i].axis("off")

axes[3].imshow(y_sample[0], cmap="gray")
axes[3].set_title("mask k")
axes[3].axis("off")
plt.suptitle(f"{case_id} | sample {sample_idx}")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 7))
plt.imshow(x_sample[1], cmap="gray")
if y_sample[0].max() > 0:
    plt.contour(y_sample[0], levels=[0.5], linewidths=1)
plt.title("Mask overlay on center slice")
plt.axis("off")
plt.show()


## Flatten case-level `.npy` arrays into sample-level index

In [ ]:
def build_sample_index(manifest_subset: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for _, row in manifest_subset.iterrows():
        x = np.load(row["input_path"], mmap_mode="r")
        y = np.load(row["mask_path"], mmap_mode="r")
        positive = np.sum(y > 0, axis=(1,2,3)) > 0

        for sample_idx in range(x.shape[0]):
            rows.append({
                "case_id": row["case_id"],
                "split": row["split"],
                "input_path": row["input_path"],
                "mask_path": row["mask_path"],
                "sample_idx": sample_idx,
                "has_cac": int(positive[sample_idx]),
            })

    return pd.DataFrame(rows)

train_index_df = build_sample_index(manifest_df[manifest_df["split"] == "train"])
val_index_df = build_sample_index(manifest_df[manifest_df["split"] == "val"])
test_index_df = build_sample_index(manifest_df[manifest_df["split"] == "test"])

print("Train:", len(train_index_df))
print("Val:", len(val_index_df))
print("Test:", len(test_index_df))
print("\nTrain label ratio:")
print(train_index_df["has_cac"].value_counts(normalize=True))


## PyTorch Dataset

In [ ]:
class CAC2p5DSegmentationDataset(Dataset):
    def __init__(self, index_df: pd.DataFrame):
        self.index_df = index_df.reset_index(drop=True)
        self._input_cache = {}
        self._mask_cache = {}

    def __len__(self):
        return len(self.index_df)

    def _load_case(self, input_path: str, mask_path: str):
        if input_path not in self._input_cache:
            self._input_cache[input_path] = np.load(input_path, mmap_mode="r")
        if mask_path not in self._mask_cache:
            self._mask_cache[mask_path] = np.load(mask_path, mmap_mode="r")
        return self._input_cache[input_path], self._mask_cache[mask_path]

    def __getitem__(self, idx):
        row = self.index_df.iloc[idx]
        x_case, y_case = self._load_case(row["input_path"], row["mask_path"])

        x = np.array(x_case[int(row["sample_idx"])], dtype=np.float32)
        y = np.array(y_case[int(row["sample_idx"])], dtype=np.float32)
        y = (y > 0).astype(np.float32)

        return {
            "image": torch.from_numpy(x),
            "mask": torch.from_numpy(y),
            "case_id": row["case_id"],
            "sample_idx": int(row["sample_idx"]),
            "has_cac": int(row["has_cac"]),
        }

train_ds = CAC2p5DSegmentationDataset(train_index_df)
val_ds = CAC2p5DSegmentationDataset(val_index_df)
test_ds = CAC2p5DSegmentationDataset(test_index_df)

print(len(train_ds), len(val_ds), len(test_ds))


## Weighted sampler for lack of positive masks

Use this for **training only**. Validation/test remain naturally distributed.


In [ ]:
def make_weighted_sampler(index_df: pd.DataFrame, positive_weight: float = 4.0):
    weights = np.where(index_df["has_cac"].values == 1, positive_weight, 1.0).astype(np.float64)
    return WeightedRandomSampler(
        weights=torch.from_numpy(weights),
        num_samples=len(weights),
        replacement=True,
    )

train_sampler = make_weighted_sampler(train_index_df, positive_weight=4.0)

train_loader = DataLoader(
    train_ds,
    batch_size=cfg.BATCH_SIZE,
    sampler=train_sampler,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=cfg.PIN_MEMORY,
)

val_loader = DataLoader(
    val_ds,
    batch_size=cfg.BATCH_SIZE,
    shuffle=False,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=cfg.PIN_MEMORY,
)

test_loader = DataLoader(
    test_ds,
    batch_size=cfg.BATCH_SIZE,
    shuffle=False,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=cfg.PIN_MEMORY,
)

batch = next(iter(train_loader))
print("Images:", batch["image"].shape)
print("Masks:", batch["mask"].shape)
print("Positive masks in sampled batch:", batch["has_cac"].sum().item())


# Model section

Replace the placeholder with your real architecture.

Model contract:

```text
Input  : [B, 3, H, W]
Output : [B, 1, H, W] logits
```

Do **not** apply sigmoid inside the model. Use logits for `BCEWithLogitsLoss`.


In [ ]:
class PlaceholderSegModel(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(16, 16, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(16, out_channels, kernel_size=1),
        )

    def forward(self, x):
        return self.net(x)

model = PlaceholderSegModel().to(DEVICE)

with torch.no_grad():
    dummy_logits = model(batch["image"].to(DEVICE))

print("Input:", batch["image"].shape)
print("Output:", dummy_logits.shape)


## Loss functions

Baseline: **Dice + BCEWithLogitsLoss**.


In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        targets = targets.float()
        dims = (1, 2, 3)

        intersection = torch.sum(probs * targets, dims)
        denominator = torch.sum(probs, dims) + torch.sum(targets, dims)
        dice = (2 * intersection + self.smooth) / (denominator + self.smooth)
        return 1 - dice.mean()

class DiceBCELoss(nn.Module):
    def __init__(self, dice_weight=0.5, bce_weight=0.5):
        super().__init__()
        self.dice = DiceLoss()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice_weight = dice_weight
        self.bce_weight = bce_weight

    def forward(self, logits, targets):
        return self.dice_weight * self.dice(logits, targets) + self.bce_weight * self.bce(logits, targets)

criterion = DiceBCELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)


## segmentation metrics

Implemented:
- Dice
- IoU
- Precision
- Recall
- Focal Tversky Loss

Add later:
- HD95
- lesion-level sensitivity

In [ ]:
def compute_metrics(logits, targets, threshold=0.5, eps=1e-6):
    probs = torch.sigmoid(logits)
    preds = (probs >= threshold).float()
    targets = targets.float()

    dims = (1, 2, 3)
    tp = torch.sum(preds * targets, dims)
    fp = torch.sum(preds * (1 - targets), dims)
    fn = torch.sum((1 - preds) * targets, dims)

    dice = (2 * tp + eps) / (2 * tp + fp + fn + eps)
    iou = (tp + eps) / (tp + fp + fn + eps)
    precision = (tp + eps) / (tp + fp + eps)
    recall = (tp + eps) / (tp + fn + eps)

    return {
        "dice": dice.mean().item(),
        "iou": iou.mean().item(),
        "precision": precision.mean().item(),
        "recall": recall.mean().item(),
    }


## Train/validation loop skeleton

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    metrics_store = {"dice": [], "iou": [], "precision": [], "recall": []}

    for batch in loader:
        x = batch["image"].to(device, non_blocking=True)
        y = batch["mask"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x.size(0)
        metrics = compute_metrics(logits.detach(), y, threshold=cfg.MASK_THRESHOLD)
        for k, v in metrics.items():
            metrics_store[k].append(v)

    epoch_loss = running_loss / len(loader.dataset)
    epoch_metrics = {k: float(np.mean(v)) for k, v in metrics_store.items()}
    return epoch_loss, epoch_metrics


@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    metrics_store = {"dice": [], "iou": [], "precision": [], "recall": []}

    for batch in loader:
        x = batch["image"].to(device, non_blocking=True)
        y = batch["mask"].to(device, non_blocking=True)

        logits = model(x)
        loss = criterion(logits, y)

        running_loss += loss.item() * x.size(0)
        metrics = compute_metrics(logits, y, threshold=cfg.MASK_THRESHOLD)
        for k, v in metrics.items():
            metrics_store[k].append(v)

    epoch_loss = running_loss / len(loader.dataset)
    epoch_metrics = {k: float(np.mean(v)) for k, v in metrics_store.items()}
    return epoch_loss, epoch_metrics


In [ ]:
history = []
best_val_dice = -1.0
best_model_path = "best_cac_2p5d_model.pt"

for epoch in range(1, cfg.EPOCHS + 1):
    start = time.time()

    train_loss, train_metrics = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss, val_metrics = validate_one_epoch(model, val_loader, criterion, DEVICE)

    elapsed = time.time() - start

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_dice": train_metrics["dice"],
        "val_dice": val_metrics["dice"],
        "train_iou": train_metrics["iou"],
        "val_iou": val_metrics["iou"],
        "train_precision": train_metrics["precision"],
        "val_precision": val_metrics["precision"],
        "train_recall": train_metrics["recall"],
        "val_recall": val_metrics["recall"],
        "seconds": elapsed,
    }
    history.append(row)

    if val_metrics["dice"] > best_val_dice:
        best_val_dice = val_metrics["dice"]
        torch.save(model.state_dict(), best_model_path)

    print(
        f"Epoch {epoch:03d}/{cfg.EPOCHS} | "
        f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
        f"train_dice={train_metrics['dice']:.4f} | val_dice={val_metrics['dice']:.4f} | "
        f"val_precision={val_metrics['precision']:.4f} | val_recall={val_metrics['recall']:.4f} | "
        f"{elapsed:.1f}s"
    )

history_df = pd.DataFrame(history)
history_df.tail()


In [ ]:
if len(history_df) > 0:
    plt.figure(figsize=(8, 5))
    plt.plot(history_df["epoch"], history_df["train_loss"], label="train loss")
    plt.plot(history_df["epoch"], history_df["val_loss"], label="val loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Loss curves")
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(history_df["epoch"], history_df["train_dice"], label="train Dice")
    plt.plot(history_df["epoch"], history_df["val_dice"], label="val Dice")
    plt.xlabel("Epoch")
    plt.ylabel("Dice")
    plt.title("Dice curves")
    plt.legend()
    plt.grid(True)
    plt.show()


## Test evaluation

Only run this after architecture decisions are finalized.


In [ ]:
if Path(best_model_path).exists():
    model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
    test_loss, test_metrics = validate_one_epoch(model, test_loader, criterion, DEVICE)

    print("Test loss:", round(test_loss, 4))
    print("Test metrics:")
    for k, v in test_metrics.items():
        print(f"{k}: {v:.4f}")
else:
    print("No trained checkpoint found.")


## Qualitative prediction display

Inspect false positives and false negatives. A metric without visual review is not enough in medical segmentation.

In [ ]:
@torch.no_grad()
def visualize_prediction(model, dataset, idx=0, threshold=0.5):
    model.eval()
    item = dataset[idx]

    x = item["image"].unsqueeze(0).to(DEVICE)
    gt = item["mask"][0].cpu().numpy()

    logits = model(x)
    prob = torch.sigmoid(logits)[0, 0].cpu().numpy()
    pred = (prob >= threshold).astype(np.float32)

    center = item["image"][1].cpu().numpy()

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    axes[0].imshow(center, cmap="gray")
    axes[0].set_title("Center slice")
    axes[0].axis("off")

    axes[1].imshow(gt, cmap="gray")
    axes[1].set_title("Ground truth")
    axes[1].axis("off")

    axes[2].imshow(pred, cmap="gray")
    axes[2].set_title("Prediction")
    axes[2].axis("off")

    axes[3].imshow(center, cmap="gray")
    if gt.max() > 0:
        axes[3].contour(gt, levels=[0.5], linewidths=1)
    if pred.max() > 0:
        axes[3].contour(pred, levels=[0.5], linewidths=1)
    axes[3].set_title("Overlay")
    axes[3].axis("off")

    plt.suptitle(f"Case {item['case_id']} | sample {item['sample_idx']} | has_cac={item['has_cac']}")
    plt.tight_layout()
    plt.show()

# Run after training:
# visualize_prediction(model, test_ds, idx=0, threshold=cfg.MASK_THRESHOLD)


# Summary

Creating a baseline U-Net
Later add more techniques like Residual + dilated bottleneck variant, HD95 metric, Lesion-level sensitivity